In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce_orders.02_silver;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders.`02_silver`.order_items AS
SELECT order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value FROM (
  SELECT *,row_number() OVER (PARTITION BY order_id, product_id ORDER BY order_item_id DESC) as rn
  FROM ecommerce_orders.`01_bronze`.order_items
) WHERE rn = 1;

In [0]:
%sql
SELECT * FROM ecommerce_orders.`02_silver`.order_items;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders.`02_silver`.order_reviews AS 
SELECT review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp FROM(
  SELECT *,row_number() OVER (PARTITION BY review_id,order_id ORDER BY review_creation_date ) AS rn
  FROM ecommerce_orders.`01_bronze`.order_reviews
)WHERE rn=1;

In [0]:
%sql
SELECT * FROM ecommerce_orders.`02_silver`.order_reviews;

In [0]:
%sql
SELECT * FROM ecommerce_orders.`01_bronze`.customers WHERE customer_city=null OR customer_state=null;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders.`02_silver`.order_reviews AS
SELECT review_id,order_id,review_score,review_comment_title,review_comment_message,date_format(review_creation_date, 'yyyy-MM-dd HH:mm:ss') AS review_creation_date,date_format(review_answer_timestamp, 'yyyy-mm-dd HH:mm:ss') AS review_answer_timestamp FROM ecommerce_orders.`01_bronze`.order_reviews;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders.`02_silver`.order_reviews AS
SELECT review_id,order_id,review_score,

CASE WHEN review_comment_title='nan' 
THEN "No Title" ELSE review_comment_title 
END AS review_comment_title,

CASE WHEN review_comment_message='nan' 
THEN "No Message" ELSE review_comment_message 
END AS review_comment_message,
date_format(review_creation_date, 'yyyy-MM-dd') AS review_creation_date,
date_format(review_answer_timestamp, 'yyyy-mm-dd HH:mm:ss') AS review_answer_timestamp 
FROM ecommerce_orders.`01_bronze`.order_reviews;

In [0]:
%sql
SELECT * FROM ecommerce_orders.`02_silver`.order_reviews;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders.`02_silver`.order_payments AS 
SELECT * FROM (
  SELECT order_id,collect_set(payment_type) as payment_types,ROUND(SUM(payment_value), 2) AS payment_value
  FROM ecommerce_orders.`01_bronze`.order_payments
  GROUP BY order_id
);

In [0]:
%sql
SELECT * FROM ecommerce_orders.`02_silver`.order_payments;
    

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders.02_silver.products AS
SELECT 
    product_id,
    CASE 
        WHEN product_category_name IS NULL OR product_category_name = '' THEN 'Unknown'
        ELSE product_category_name
    END AS product_category_name,
    COALESCE(product_name_lenght, '0') as product_name_length,
    COALESCE(product_description_lenght, '0') as product_description_length,
    COALESCE(product_photos_qty, '0') as product_photos_qty,
    COALESCE(product_weight_g, '0') as product_weight_g,
    COALESCE(product_length_cm, '0') as product_length_cm,
    COALESCE(product_height_cm, '0') as product_height_cm,
    COALESCE(product_width_cm, '0') as product_width_cm
FROM ecommerce_orders.`01_bronze`.products;

In [0]:
%sql
SELECT * FROM ecommerce_orders.`02_silver`.products;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders.`02_silver`.orders AS
SELECT 
    order_id,
    COALESCE(customer_id, 'Unknown') AS customer_id,
    COALESCE(order_status, 'Unknown') AS order_status,
    COALESCE(order_purchase_timestamp,  'Unknown') AS order_purchase_timestamp,
    COALESCE(order_approved_at,  'Unknown') AS order_approved_at,
    COALESCE(order_delivered_carrier_date,  'Unknown') AS order_delivered_carrier_date,
    COALESCE(order_delivered_customer_date,  'Unknown') AS order_delivered_customer_date,
    COALESCE(order_estimated_delivery_date,  'Unknown') AS order_estimated_delivery_date,

    CASE 
        WHEN order_delivered_customer_date = 'Unknown' OR order_purchase_timestamp = 'Unknown' THEN 0
        WHEN CAST(order_delivered_customer_date AS DATE) < CAST(order_purchase_timestamp AS DATE) THEN 1
        ELSE 0
    END AS is_invalid_delivery,
    
    CASE 
        WHEN order_delivered_customer_date = 'Unknown' OR order_estimated_delivery_date = 'Unknown' THEN 0
        WHEN CAST(order_delivered_customer_date AS DATE) > CAST(order_estimated_delivery_date AS DATE) THEN 1
        ELSE 0
    END AS is_late_delivery,
    
    COALESCE(
    CASE 
        WHEN order_delivered_customer_date = 'Unknown' OR order_purchase_timestamp = 'Unknown' THEN 0
        ELSE CAST(DATEDIFF(
            CAST(order_delivered_customer_date AS DATE), 
            CAST(order_purchase_timestamp AS DATE)
        ) AS INTEGER)
    END,
0) AS delivery_days

FROM ecommerce_orders.`01_bronze`.orders;

In [0]:
%sql
DROP TABLE ecommerce_orders.`02_silver`.orders

In [0]:
%sql
SELECT * FROM ecommerce_orders.`02_silver`.orders;